# 04 - Diagnostics and calibration

Learning dynamics of a run, the direction heads' spread and bias, and an interactive
**calibration explorer**: refit the calibration pipeline on the calibration block with another
interval scale, with or without delta shrinkage, at another miscoverage level, and see test
coverage, interval width, temperature, delta beta, reliability and coverage over time.

In [1]:
# Parameters
RUN_DIR = None
RUNS_DIR = "../runs"
CSV_PATH = "../binance_btcusdt_1min_ccxt.csv"

In [2]:
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display

from neural_trade.evaluation.frame import HORIZONS
from neural_trade.metrics.direction_labels import direction_labels_np
from neural_trade.notebook import CalibrationExplorer, load_run_blocks, pick_run
from neural_trade.telemetry.epoch_logger import read_metrics

run_dir = pick_run(RUN_DIR, RUNS_DIR)   # newest run with a serving bundle, or a clear error
print("run:", run_dir)
log = pd.DataFrame(read_metrics(run_dir / "metrics.jsonl"))
blocks = load_run_blocks(run_dir, csv_path=CSV_PATH)

run: ..\runs\20260924T114819Z-e23fd9f-dirty-af67ee43
CalibrationPipeline loaded from '..\runs\20260924T114819Z-e23fd9f-dirty-af67ee43\artifacts\calibration/'


Dataset length after cleaning: 43500


Date range after cleaning: 2025-10-11 02:30:00+00:00 to 2025-11-10 07:29:00+00:00


## Learning dynamics

In [3]:
cols = [c for c in ("loss", "val_loss", "val_nll_loss", "val_crps_loss", "val_dir_mcc_h1", "val_gauss_dir_mcc_h1",
                    "val_pred_up_rate_h1", "nonfinite_grad_steps", "grad_global_norm") if c in log]
px.line(log, x="epoch", y=cols, facet_col="variable", facet_col_wrap=3, height=600).update_yaxes(matches=None).show()

## Direction heads on the test block (served = calibrated)

In [4]:
test, cfg = blocks["test"], blocks["config"]
labels = direction_labels_np(test.y, test.last_close, cfg.DIR_DEADBAND_BPS)
pd.DataFrame({h: {"P(up) std": test.direction_prob[h].std(), "P(up) mean": test.direction_prob[h].mean(),
                  "true up rate (outside deadband)": labels[h][0][labels[h][1]].mean(),
                  "share outside deadband": labels[h][1].mean(),
                  "served delta std ($)": test.delta[h].std(), "realised std ($)": test.y[:, i].std()}
              for i, h in enumerate(HORIZONS)})

,h0,h1,h2
P(up) std,0.046319,0.062945,0.049320
P(up) mean,0.508502,0.482673,0.511313
true up rate (outside deadband),0.513938,0.514657,0.518897
share outside deadband,0.738668,0.787313,0.811774
served delta std ($),0.000000,0.000000,4.932467
realised std ($),196.112845,235.915870,268.822656


## Calibration explorer

Fit on the calibration block, scored on the test block; the run itself is not changed.

In [5]:
calib = CalibrationExplorer(blocks)
display(calib.widget())
calib.click_refit()   # the run's saved settings first; then change them and press Refit

Static copy of that refit (the explorer above stays interactive):

In [6]:
display(calib.last_table.round(4))
reliability, coverage = calib.figures("h1")
reliability.show()
coverage.show()

,temperature,delta beta,coverage,target,mean width $,EV raw delta,EV served delta,ECE raw,ECE calibrated
h0,1.4631,0.0000,0.9027,0.9,643.1248,-0.0142,0.0000,0.0396,0.0276
h1,1.8697,0.0000,0.9059,0.9,781.3664,-0.0739,0.0000,0.0599,0.0379
h2,1.5044,0.0608,0.9122,0.9,913.5853,-0.0796,0.0004,0.0383,0.0240
